# Diagnóstico de ingestão

## Multiline
- Testados `multiLine=False` e `multiLine=True` nos 8 CSVs.
- As contagens de linhas, quantidades de colunas e amostras foram iguais nos dois modos.
- **Decisão:** usar `multiLine=False` por padrão.
- **Limitação:** os testes não comprovam a ausência de campos com quebras de linha entre aspas em todo o conteúdo.

## Encoding

| Arquivo | Encoding definido | Evidência |
|---|---|---|
| `Chamados_Hora.CSV` | `windows-1252` | UTF-8 inválido; caracteres acentuados corrompidos em UTF-8 |
| `base_atendentes.csv` | `UTF-8` | UTF-8 válido; nomes acentuados corretos |
| `base_motivos.csv` | `UTF-8` | UTF-8 válido; motivos acentuados corretos |
| `canais.csv` | `UTF-8` | Conteúdo ASCII puro |
| `chamados.csv` | `UTF-8` | UTF-8 válido; texto correto em português |
| `clientes.csv` | `windows-1252` | UTF-8 inválido; nomes acentuados corretos em Windows-1252 |
| `custos.csv` | `UTF-8` | Conteúdo ASCII puro |
| `pesquisa_satisfacao.csv` | `UTF-8` | Conteúdo ASCII puro |

### Observações sobre encoding
- Nenhum dos arquivos apresentou BOM UTF-8.
- `Chamados_Hora.CSV` e `clientes.csv` não passaram na decodificação estrita UTF-8.
- Nesses dois arquivos, Windows-1252 produziu texto coerente e não apresentou caracteres de substituição (`�`).
- `base_atendentes.csv`, `base_motivos.csv` e `chamados.csv` foram lidos corretamente em UTF-8; Windows-1252 produziu texto corrompido.
- `canais.csv`, `custos.csv` e `pesquisa_satisfacao.csv` contêm apenas bytes ASCII. Não é possível determinar a codificação original a partir desses bytes; UTF-8 foi escolhido por convenção operacional.
- Para os arquivos incompatíveis com UTF-8, não foi possível distinguir Windows-1252 de ISO-8859-1 usando os bytes observados. Windows-1252 foi escolhido por convenção operacional e pela coerência dos textos resultantes.


In [0]:

# Configurações
catalog = "v_credit_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Schemas completos
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Caminho da landing
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

# Caminhos dos arquivos
path_chamados_hora = f"{landing_path}/Chamados_Hora.CSV"
path_atendentes = f"{landing_path}/base_atendentes.csv"
path_motivos = f"{landing_path}/base_motivos.csv"
path_canais = f"{landing_path}/canais.csv"
path_chamados = f"{landing_path}/chamados.csv"
path_clientes = f"{landing_path}/clientes.csv"
path_custos = f"{landing_path}/custos.csv"
path_pesquisa_satisfacao = f"{landing_path}/pesquisa_satisfacao.csv"

# Determinados com base nas analises de `csv_ingestion_diagnostics`
encodings = {
    "Chamados_Hora.CSV": "windows-1252",
    "base_atendentes.csv": "UTF-8",
    "base_motivos.csv": "UTF-8",
    "canais.csv": "UTF-8",
    "chamados.csv": "UTF-8",
    "clientes.csv": "windows-1252",
    "custos.csv": "UTF-8",
    "pesquisa_satisfacao.csv": "UTF-8",
}




In [0]:

# Arquivos da landing
files = sorted(
    f.path
    for f in dbutils.fs.ls(landing_path)
    if f.name.lower().endswith(".csv")
)

for path in files:
    print(path.split("/")[-1])


# Comparação de Leitura

In [0]:

# Testa leitura padrão e multiline
for path in files:
    print(f"\n{'=' * 60}\n{path.split('/')[-1]}")

    for multiline in [False, True]:
        df = (
            spark.read
            .format("csv")
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", multiline)
            .option("mode", "PERMISSIVE")
            .load(path)
        )

        print(f"multiline={multiline} | linhas={df.count()} | colunas={len(df.columns)}")
        print(df.columns)


# Teste de encoding

In [0]:

for path in files:
    print(f"\n{'=' * 50}\n{path.split('/')[-1]}")

    for encoding in ["UTF-8", "windows-1252"]:
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("encoding", encoding)
            .load(path)
        )

        print(f"\n[{encoding}]")
        df.show(3, truncate=False)


In [0]:

from pyspark.sql import functions as F

for path in files:
    nome = path.split("/")[-1]
    print(f"\n{'=' * 50}\n{nome}")

    resultados = {}

    for encoding in ["UTF-8", "windows-1252"]:
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("encoding", encoding)
            .option("mode", "PERMISSIVE")
            .load(path)
        )

        # Procura linhas com caracteres fora do ASCII
        condicoes = [
            F.col(c).rlike(r"[^\x00-\x7F]")
            for c in df.columns
        ]

        tem_nao_ascii = condicoes[0]
        for condicao in condicoes[1:]:
            tem_nao_ascii = tem_nao_ascii | condicao

        amostra = (
            df.filter(tem_nao_ascii)
            .limit(5)
            .collect()
        )

        resultados[encoding] = amostra

        print(f"\n[{encoding}]")
        if amostra:
            for linha in amostra:
                print(linha)
        else:
            print("Nenhuma linha com caracteres não ASCII na amostra.")



In [0]:

import csv
import io

for path in files:
    nome = path.split("/")[-1]

    raw = (
        spark.read.format("binaryFile")
        .load(path)
        .select("content")
        .first()["content"]
    )

    print(f"\n{'=' * 50}\n{nome}")

    for encoding in ["UTF-8", "windows-1252"]:
        texto = raw.decode(encoding, errors="replace")
        substituicoes = texto.count("\uFFFD")

        exemplos = []
        leitor = csv.reader(io.StringIO(texto))

        try:
            next(leitor, None)  # Cabeçalho

            for linha in leitor:
                if any(
                    any(ord(c) > 127 for c in campo)
                    for campo in linha
                ):
                    exemplos.append(linha)
                    if len(exemplos) >= 3:
                        break
        except csv.Error as e:
            print(f"[{encoding}] Erro no CSV: {e}")
            continue

        print(f"\n[{encoding}]")
        print(f"Caracteres �: {substituicoes:,}")

        for linha in exemplos:
            print(linha)

        if not exemplos:
            print("Nenhuma linha com caracteres não ASCII encontrada.")


# Inspeção dos bytes originais

In [0]:

for path in files:
    nome = path.split("/")[-1]

    raw = (
        spark.read.format("binaryFile")
        .load(path)
        .select("content")
        .first()["content"]
    )

    try:
        raw.decode("utf-8", errors="strict")
        utf8_valido = True
    except UnicodeDecodeError:
        utf8_valido = False

    print(f"\n{nome}")
    print(f"Tamanho: {len(raw):,} bytes")
    print(f"UTF-8 válido: {utf8_valido}")
    print(f"Bytes >= 0x80: {sum(b >= 128 for b in raw):,}")
    print(f"Bytes 0x80–0x9F: {sum(0x80 <= b <= 0x9F for b in raw):,}")
    print(f"BOM UTF-8: {raw.startswith(bytes([0xEF, 0xBB, 0xBF]))}")


# Analisando a viabilidade do uso de MERGE

In [0]:
# Leitura dos dados brutos
def ler_csv(path, encoding):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", encoding)
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", False)
        .csv(path)
    )

df_chamados_hora_raw = ler_csv(path_chamados_hora, encodings["Chamados_Hora.CSV"])
df_atendentes_raw = ler_csv(path_atendentes, encodings["base_atendentes.csv"])
df_motivos_raw = ler_csv(path_motivos, encodings["base_motivos.csv"])
df_canais_raw = ler_csv(path_canais, encodings["canais.csv"])
df_chamados_raw = ler_csv(path_chamados, encodings["chamados.csv"])
df_clientes_raw = ler_csv(path_clientes, encodings["clientes.csv"])
df_custos_raw = ler_csv(path_custos, encodings["custos.csv"])
df_pesquisa_satisfacao_raw = ler_csv(path_pesquisa_satisfacao, encodings["pesquisa_satisfacao.csv"])

dataframes = {
    "Chamados_Hora": df_chamados_hora_raw,
    "Atendentes": df_atendentes_raw,
    "Motivos": df_motivos_raw,
    "Canais": df_canais_raw,
    "Chamados": df_chamados_raw,
    "Clientes": df_clientes_raw,
    "Custos": df_custos_raw,
    "Pesquisa_Satisfacao": df_pesquisa_satisfacao_raw,
}

## Verificar unicidade e valores nulos por coluna

In [0]:
from pyspark.sql import functions as F

for nome, df in dataframes.items():
    total = df.count()

    print(f"\n{'=' * 50}\n{nome}: {total:,} registros")

    for coluna in df.columns:
        resultado = df.agg(
            F.countDistinct(coluna).alias("distintos"),
            F.sum(
                F.when(
                    F.col(coluna).isNull() | (F.trim(F.col(coluna)) == ""),
                    1
                ).otherwise(0)
            ).alias("nulos_ou_vazios")
        ).first()

        distintos = resultado["distintos"]
        nulos = resultado["nulos_ou_vazios"] or 0

        unica = distintos == total and nulos == 0

        print(
            f"{coluna}: distintos={distintos:,}, "
            f"nulos/vazios={nulos:,}, "
            f"única={unica}"
        )

## Verificar combinações de duas colunas

In [0]:
from itertools import combinations

for nome, df in dataframes.items():
    total = df.count()
    print(f"\n{'=' * 50}\n{nome}")

    for c1, c2 in combinations(df.columns, 2):
        distintos = df.select(c1, c2).distinct().count()

        if distintos == total:
            nulos = df.filter(
                F.col(c1).isNull()
                | F.col(c2).isNull()
                | (F.trim(F.col(c1)) == "")
                | (F.trim(F.col(c2)) == "")
            ).count()

            if nulos == 0:
                print(f"Chave composta candidata: ({c1}, {c2})")

In [0]:
from pyspark.sql import functions as F

# 1. Custos: os registros repetidos são cópias exatas?
print("CUSTOS — duplicatas de linhas completas:")
print(df_custos_raw.count() - df_custos_raw.distinct().count())

print("\nCUSTOS — registros com id_custo repetido:")
ids_custo_repetidos = (
    df_custos_raw
    .groupBy("id_custo")
    .count()
    .filter(F.col("count") > 1)
    .select("id_custo")
)

df_custos_raw.join(
    ids_custo_repetidos, on="id_custo", how="inner"
).orderBy("id_custo").show(20, truncate=False)


# 2. Pesquisa: existem pesquisas diferentes para o mesmo chamado?
print("\nPESQUISA — amostra de chamados com mais de uma pesquisa:")
ids_chamado_repetidos = (
    df_pesquisa_satisfacao_raw
    .groupBy("id_chamado")
    .count()
    .filter(F.col("count") > 1)
    .select("id_chamado")
)

df_pesquisa_satisfacao_raw.join(
    ids_chamado_repetidos,
    on="id_chamado",
    how="inner"
).orderBy("id_chamado").show(20, truncate=False)


# 3. Chamados: os IDs correspondem entre os dois arquivos?
ids_hora = df_chamados_hora_raw.select(
    F.col("ID_Chamado").alias("id_chamado")
).distinct()

ids_chamados = df_chamados_raw.select("id_chamado").distinct()

print("\nCOMPARAÇÃO DE IDs ENTRE CHAMADOS E CHAMADOS_HORA:")
print(
    "IDs em Chamados que não estão em Chamados_Hora:",
    ids_chamados.join(ids_hora, "id_chamado", "left_anti").count()
)
print(
    "IDs em Chamados_Hora que não estão em Chamados:",
    ids_hora.join(ids_chamados, "id_chamado", "left_anti").count()
)